# Phase 2 on a TPU v5e — `fused_gated_mlp`

The first hardware run in this repo. Four things are being measured, in the order
they stop mattering if the runtime dies:

1. **Correctness on device.** Interpret mode is not proof of hardware correctness
   ([jax#36287](https://github.com/jax-ml/jax/issues/36287)), so nothing here is
   publishable until cell 2 is green.
2. **The VMEM budget a `pallas_call` actually gets** — the default of
   `--xla_tpu_scoped_vmem_limit_kib`, which is not published anywhere and which
   `pltpu.InterpretParams` cannot be asked, because it models no VMEM capacity.
3. **Where the kernel sits against the roofline**, with XLA alongside.
4. **Whether copy elision is contractual on the hardware path**, from a DMA count
   in an xprof trace.

Every cell is independently re-runnable and writes its artifact before the next
one starts. Artifacts live on the runtime's local disk and section 8 packages
them for download, so pull them down before the session ends. No cell should take
more than about three minutes.

## The predictions, registered before the run

The byte model is the corrected one. The grid is
`(tokens // block_t, hidden_dim // block_h)` with the hidden axis **innermost**,
and the `w_*` `index_map`s vary in the inner index — so one `t` step walks every
hidden block, and when the inner index resets, block 0 is no longer the
previously fetched slice. Copy elision skips only *consecutive* identical
slices, so the weights are re-read **once per `t` step**:

```
bytes = (tokens // block_t) · 3·E·H·dtype  +  2·T·E·dtype
```

At the tested geometry (`T=256, block_t=128`) that is 193.5 MB and intensity
**63**, not 97.9 MB and 125. Both counts are in `shapes.mlp_bytes`
(`elide_weights=True` gives the one-pass counterfactual); cell 6 adjudicates.

v5e publishes one compute roof, for bf16, and TPU emulates an fp32 matmul with
3 or 6 bf16 passes. So there are three candidate roofs, three ridges, and two
crossovers that a timing sweep can actually resolve:

| `precision` | passes | peak | ridge | HBM-bound at | at the ridge | compute-bound at |
|---|---|---|---|---|---|---|
| `DEFAULT` | 1 | 197 TFLOP/s | 240.5 | 128, 256 | **512** (+1.4%) | 1024, 2048 |
| `HIGH` | 3 | 65.7 TFLOP/s | 80.2 | **128** (−21%) | — | 256, 512, 1024, 2048 |
| `HIGHEST` | 6 | 32.8 TFLOP/s | 40.1 | — | — | every T |

`DEFAULT` at T=512 is registered **in advance as unresolvable**: 1.4% is inside
the noise of any timing collected here, so it is not scored either way. `HIGH`'s
crossover between T=128 and T=256 would have been the better test of the byte
model, because its margin is 21%.

### Amendment: `HIGH` is not reachable from a Pallas kernel on jax 0.11.0

Registered before any timing was collected, on the first attempt to run cell 3.
The Mosaic lowering rule for `dot_general` accepts `DEFAULT` and `HIGHEST` only;
anything else raises, and `HIGH` lands in the `else`:

```
NotImplementedError: Unsupported dot precision: HIGH
  jax/_src/pallas/mosaic/lowering.py:2879, in _dot_general_lowering_rule
```

The `bf16x3` branch that would accept it —

```python
elif precision == lax.Precision.HIGH:
    precision_attr = ir.Attribute.parse("#tpu.contract_precision<bf16x3>")
```

— exists on `jax` `main` and in no release: 0.11.0 is the newest on PyPI, and the
tagged `jax-v0.11.0` file raises at exactly the line in the traceback. The XLA
path is unaffected, because `jax.default_matmul_precision` reaches XLA rather
than Mosaic, but a kernel row and an XLA row are only comparable at the same
precision, so the sweep drops `HIGH` on both.

Two consequences, both taken rather than worked around:

* The kernel is probed and swept at `DEFAULT` and `HIGHEST`. The `HIGH` row above
  stays in the table as a *prediction that could not be tested on this runtime* —
  not as one that failed.
* The 21%-margin crossover is gone, and `HIGHEST` predicts compute-bound at every
  `T`, so **no** timing crossover is resolvable this run. Cell 6's DMA count is
  therefore the byte model's only adjudicator here, rather than its corroborator.

**Second prediction:** the on-device error against `reference.gated_mlp` will be
far larger than the CPU run's 3.28e-6 at `DEFAULT`, because the multiplies are
bf16. `GEMMA_RTOL = 1e-4` is a CPU measurement and is expected to fail here. The
notebook **reports** the error rather than asserting an inherited tolerance.

## 0 · Preflight

Refuses to go on without a TPU, and prints the hardware constants everything
below is measured against. `tpu_info` is the only primary source found that
states v5e VMEM; `docs.cloud.google.com/tpu/docs/v5e` does not.

In [ ]:
# The Colab TPU runtime ships jax 0.7.2, which has no
# jax._src.pallas.mosaic.tpu_info. Upgrade, then restart the session.
%pip install -q -U "jax[tpu]>=0.11.0"

import jax

# The upgrade lands on disk but not in a kernel that already imported jax, so
# say so plainly rather than failing on the tpu_info import three lines down.
# Compared as integers: "0.7.2" sorts *after* "0.11.0" as a string.
installed = tuple(int(part) for part in jax.__version__.split(".")[:2])
assert installed >= (0, 11), (
    f"jax is still {jax.__version__} in this kernel. Runtime > Restart session, "
    "then re-run this cell. The pip line above only takes effect after a restart."
)

from jax._src.pallas.mosaic import tpu_info as ti

print("jax        ", jax.__version__)
print("devices    ", jax.devices())

platform = jax.devices()[0].platform
assert platform == "tpu", (
    f"backend is {platform!r}, not tpu. Runtime > Change runtime type > TPU. "
    "No number produced here would be publishable."
)

# The first argument must be the ChipVersion enum member; a string fails late
# and unhelpfully on `.is_lite`. Second argument is TensorCores per device.
info = ti.get_tpu_info_for_chip(ti.ChipVersion.TPU_V5E, 1)
for field in (
    "bf16_ops_per_second",
    "mem_bw_bytes_per_second",
    "vmem_capacity_bytes",
    "smem_capacity_bytes",
    "hbm_capacity_bytes",
    "num_mxus",
    "mxu_column_size",
    "num_lanes",
    "num_sublanes",
):
    print(f"{field:26} {getattr(info, field):>20,}")

print()
print(f"VMEM {info.vmem_capacity_bytes / 1024**2:.0f} MiB is the *physical* capacity.")
print("What a pallas_call is allowed to use is the scoped-vmem limit -- cell 4.")

## 1 · Setup

Clone and install the package only — `--no-deps`, so cell 0's jax stays exactly
as it is.

Everything below writes to `/content`, which dies with the runtime. Each cell
saves its artifact as soon as it has one, so an interrupted session still leaves
the completed sweeps on disk; section 8 zips them for download.

In [ ]:
import pathlib
import sys

RESULTS = pathlib.Path("/content/results")
TRACES = pathlib.Path("/content/traces")
for directory in (RESULTS, TRACES):
    directory.mkdir(parents=True, exist_ok=True)

REPO = pathlib.Path("/content/gemma3-tpu-pallas")
if not REPO.exists():
    !git clone --depth 1 https://github.com/denis-mil/gemma3-tpu-pallas {REPO}

%pip install -q -e {REPO} --no-deps

# Belt and braces: makes the cell re-runnable without a kernel restart if the
# editable install has not yet landed on sys.path.
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import gemma3_pallas

print("package at ", gemma3_pallas.__file__)
print("artifacts at", RESULTS)

# Artifacts left by an earlier run of this session. A re-run overwrites the file
# it produces, so anything worth keeping should be downloaded first.
existing = sorted(path for path in RESULTS.rglob("*") if path.is_file())
if existing:
    print(f"\n{len(existing)} artifact(s) already present:")
    for path in existing:
        print(f"  {path.relative_to(RESULTS)}  {path.stat().st_size / 1e3:.0f} kB")

## 2 · Correctness on device

The load-bearing cell. `interpret=False`, Gemma's real geometry, against
`reference.gated_mlp` — and it **reports** the error instead of asserting the
CPU-derived tolerance, because `DEFAULT` precision means bf16 multiplies and
1e-4 is not expected to survive that.

A timing number produced without this cell having run is not publishable. That
is the ADR-0003 contract.

In [ ]:
import json

import numpy as np
import jax.numpy as jnp

from gemma3_pallas import bench
from gemma3_pallas.mlp import fused_gated_mlp
from gemma3_pallas.reference import gated_mlp
from gemma3_pallas.shapes import GEMMA3_1B

bench.require_tpu()

# block_h = 384, not 768. At 768 the five double-buffered blocks come to
# 22.50 MiB and Mosaic refuses the kernel at the default scoped-VMEM limit:
#
#   Scoped allocation with size 22.50M and limit 16.00M exceeded scoped vmem
#   limit by 6.50M
#
# That message is cell 4's answer arriving early -- the undocumented default of
# --xla_tpu_scoped_vmem_limit_kib is 16 MiB -- and it is recorded as such rather
# than worked around silently. It also disagrees with the working-set model in
# cell 4: the two [block_t, block_h] intermediates (0.75 MiB at 128x768) are not
# in the 22.50 MiB the compiler counted, so they are not part of the same
# allocation. Correctness needs no particular block size, so this cell takes the
# largest geometry that fits (12.75 MiB by that model) and cell 4 measures the
# boundary properly.
TOKENS, BLOCK_T, BLOCK_H = 256, 128, 384

x, w_gate, w_up, w_down = bench.operands(TOKENS)
# Ground truth must not share the kernel's rounding. Jitted with no placement,
# `gated_mlp` runs on the TPU, where its bare `@` lowers at DEFAULT precision --
# bf16 multiplies. A bf16 kernel would then "agree" with it to fp32 accumulation
# noise (~1e-6) while both sat ~1e-2 from the true answer, and the agreement
# would read as accuracy. Committing the operands to the CPU puts the reference
# on a backend where fp32 multiplies are fp32, at the cost of a few seconds.
CPU = jax.devices("cpu")[0]
reference_out = np.asarray(
    jax.jit(gated_mlp)(*(jax.device_put(a, CPU) for a in (x, w_gate, w_up, w_down)))
)
scale = float(np.abs(reference_out).max())


def error_report(precision, *, block_t=BLOCK_T, block_h=BLOCK_H, vmem_limit_bytes=None):
    """Max absolute and relative error against the fp32 reference, on device.

    `vmem_limit_bytes` raises the scoped-VMEM budget for this one call, so a
    geometry that will not compile at the default can still be checked for
    correctness -- pass e.g. 64 * 1024**2 with block_h=768.
    """
    from jax.experimental.pallas import tpu as pltpu

    compiler_params = (
        None
        if vmem_limit_bytes is None
        else pltpu.CompilerParams(vmem_limit_bytes=vmem_limit_bytes)
    )
    got = np.asarray(
        fused_gated_mlp(
            x,
            w_gate,
            w_up,
            w_down,
            block_t=block_t,
            block_h=block_h,
            interpret=False,
            precision=precision,
            compiler_params=compiler_params,
        )
    )
    absolute = np.abs(got - reference_out)
    # Guarded: the output has near-zero entries, and a relative error against
    # one of those says more about the divisor than about the kernel.
    denominator = np.maximum(np.abs(reference_out), 1e-3 * scale)
    return {
        "precision": bench.precision_label(precision),
        "tokens": TOKENS,
        "block_t": block_t,
        "block_h": block_h,
        "vmem_limit_bytes": vmem_limit_bytes,
        "output_scale": scale,
        "max_abs_error": float(absolute.max()),
        "max_rel_error": float((absolute / denominator).max()),
        "cpu_reference_max_abs_error": 3.28e-6,
    }


correctness = [error_report(None)]
(RESULTS / "correctness.json").write_text(json.dumps(correctness, indent=2))

row = correctness[0]
print(f"block_t={row['block_t']}, block_h={row['block_h']}")
print(f"output scale        {row['output_scale']:.4g}")
print(f"max abs error       {row['max_abs_error']:.4g}   (CPU interpret run: 3.28e-6)")
print(f"max rel error       {row['max_rel_error']:.4g}")
print()
print("Nonzero but bounded is the expected shape of this result. A max abs error")
print("of ~1e-2 at DEFAULT is bf16 multiplies, not a bug; 0.0 would be the")
print("surprise, and a NaN or a value comparable to the output scale is a bug.")

## 3 · Precision probe

The same check at the precisions a Pallas kernel can actually be lowered at.
There is no published fp32 peak for v5e — TPU emulates an fp32 matmul with 3 or 6
bf16 passes — so the roof the kernel is measured against depends on which
precision it ran at. The error should fall roughly monotonically; the pass count
whose error approaches the CPU run's 3.28e-6 identifies the roof to use in cell 7.

`HIGH` is **not** probed: Mosaic's `dot_general` rule on jax 0.11.0 raises
`NotImplementedError: Unsupported dot precision: HIGH`, and the `bf16x3` branch
that accepts it is unreleased (see the amendment at the top). So this cell asks
`DEFAULT` versus `HIGHEST` — 1 pass versus 6 — and the 3-pass roof stays a
prediction. The probe is written so an unlowerable precision is *recorded*, with
the compiler's message, rather than killing the cell: same treatment cell 4 gives
a Mosaic VMEM failure.

In [ ]:
# "high" is absent: Mosaic on jax 0.11.0 has no lowering branch for it and
# raises at trace time. It is left in the list below, commented, so the omission
# reads as a toolchain limit rather than an oversight -- and so it can be put
# back on a jax that carries the #tpu.contract_precision<bf16x3> branch.
PROBE_PRECISIONS = ("highest",)  # ("high", "highest") once HIGH lowers


def probe(precision):
    """`error_report`, but a precision Mosaic cannot lower is a recorded result.

    A lowering refusal is a fact about the toolchain and belongs in the
    artifact; it should also not cost the rows that do lower, which is why this
    catches rather than letting the cell die.
    """
    try:
        return error_report(precision)
    except NotImplementedError as exc:
        return {
            "precision": bench.precision_label(precision),
            "tokens": TOKENS,
            "block_t": BLOCK_T,
            "block_h": BLOCK_H,
            "status": "unsupported",
            "error": str(exc),
            "jax_version": jax.__version__,
        }


for precision in PROBE_PRECISIONS:
    correctness.append(probe(precision))

(RESULTS / "correctness.json").write_text(json.dumps(correctness, indent=2))

print(f"{'precision':<10} {'max abs':>12} {'max rel':>12}   vs CPU 3.28e-6")
for row in correctness:
    if row.get("status") == "unsupported":
        print(f"{row['precision']:<10} {'not lowerable':>12}   {row['error'].splitlines()[0]}")
        continue
    ratio = row["max_abs_error"] / 3.28e-6
    print(
        f"{row['precision']:<10} {row['max_abs_error']:>12.4g} "
        f"{row['max_rel_error']:>12.4g}   {ratio:>10.1f}x"
    )

print()
print("Read this as: whichever precision lands near the fp32 noise floor of the")
print("CPU reference (3.28e-6) is the one doing full-mantissa work, and its pass")
print("count is the divisor on the 197 TFLOP/s roof. DEFAULT should sit ~1e-2 --")
print("one bf16 pass -- and HIGHEST near the floor at six. Set PASSES in section 7")
print("from this table: 1 for DEFAULT, 6 for HIGHEST. The 3-pass roof is")
print("unreachable on this jax.")

## 4 · VMEM sweep — the budget a `pallas_call` actually gets

The other load-bearing cell. `pltpu.InterpretParams` models **no** VMEM capacity,
so this question cannot be asked anywhere but here.

The method: run configs of growing scoped allocation at the default limit and
record which ones compile. A Mosaic VMEM failure is the measurement, not an
error, so `bench_kernel` records it with the compiler's message and the sweep
continues. The largest that compiles and the smallest that does not **bracket**
the default.

Two things the first run of this cell got wrong, both fixed below. A block shape
`_validate` rejects fails for reasons that have nothing to do with VMEM, so those
rows are labelled `rejected` and kept out of the bracket. And an operand whose
grid axis has a single step is allocated one buffer rather than two, which is why
`block_h=6912` fits on the chip at all. Then the first failing config is re-run with explicit
`vmem_limit_bytes` values to confirm the flag moves that boundary — which also
confirms the budget is sweepable per `pallas_call`, with no runtime restart.

In [ ]:
cfg = GEMMA3_1B


def buffers_for(steps):
    """Pallas double-buffers an operand only when its block index moves.

    A grid axis with a single step hands every visit the same block, so there is
    nothing to prefetch and Mosaic allocates one buffer instead of two. Measured:
    block_h=6912 leaves the h axis one step long and the kernel compiles inside
    128 MiB, which two 91.12 MiB weight buffers could not have.
    """
    return 2 if steps > 1 else 1


def scoped_bytes(tokens, block_t, block_h, *, cfg=cfg, dtype_bytes=4):
    """VMEM the compiler counts against `--xla_tpu_scoped_vmem_limit_kib`.

    Five blocked operands -- x, w_gate, w_up, w_down and the output -- each
    buffered according to whether its own grid axis actually loops. The two
    [block_t, block_h] intermediates are excluded: at 128x768 Mosaic reported
    22.50M against a 23.25 MiB working set, and 0.75 MiB is those two exactly.
    """
    weights = 3 * cfg.embed_dim * block_h  # w_gate, w_up, w_down
    streamed = 2 * block_t * cfg.embed_dim  # x, output
    return dtype_bytes * (
        buffers_for(cfg.hidden_dim // block_h) * weights
        + buffers_for(tokens // block_t) * streamed
    )


def working_set_bytes(tokens, block_t, block_h, *, cfg=cfg, dtype_bytes=4):
    """`scoped_bytes` plus the two fp32 intermediates the kernel body holds."""
    scoped = scoped_bytes(tokens, block_t, block_h, cfg=cfg, dtype_bytes=dtype_bytes)
    return scoped + dtype_bytes * 2 * block_t * block_h


# `_validate` requires block_h to divide hidden_dim (6912) and to be a multiple
# of the 128-lane width, so 512 and 1728 are not candidates at all: 6912 % 512
# is 256, and 1728 % 128 is 64. Such a rejection is a shape error, not a VMEM
# measurement, and the two are kept apart below. What survives has nothing
# between 384 and 768, so no choice of block narrows the bracket around the
# 16 MiB limit -- the compiler's own message is what names it.
BLOCK_HS = [128, 256, 384, 768, 1152, 2304, 3456, 6912]
BLOCK_TS = [128, 256]

vmem_specs = [
    dict(tokens=TOKENS, block_t=bt, block_h=bh, precision=None)
    for bt in BLOCK_TS
    for bh in BLOCK_HS
]
vmem_specs.sort(key=lambda s: scoped_bytes(s["tokens"], s["block_t"], s["block_h"]))

vmem_results = []
vmem_path = RESULTS / "vmem_sweep.json"


def failure_kind(result):
    """One of "ok", "rejected" (bad shape) or "oom" (the budget refused it)."""
    if result.status == "ok":
        return "ok"
    return "rejected" if result.error.startswith("ValueError") else "oom"


def keep_vmem(result):
    vmem_results.append(result)
    bench.save(vmem_results, vmem_path)  # rewritten in full after every config
    mib = scoped_bytes(result.tokens, result.block_t, result.block_h) / 1024**2
    median = result.median()
    timing = f"{1e3 * median:8.3f} ms" if median else "        --"
    print(f"block_t={result.block_t:<4} block_h={result.block_h:<5} "
          f"scoped {mib:7.2f} MiB  {failure_kind(result):<9} {timing}")


bench.sweep(vmem_specs, interpret=False, on_result=keep_vmem)


def asked(result):
    """Scoped VMEM this configuration asked for, as the sort key for extremes."""
    return scoped_bytes(result.tokens, result.block_t, result.block_h)


ok = [r for r in vmem_results if r.status == "ok"]
oom = [r for r in vmem_results if failure_kind(r) == "oom"]

print()
if ok:
    largest = max(ok, key=asked)
    print(f"largest that compiled:  {asked(largest) / 1024**2:.2f} MiB "
          f"(block_t={largest.block_t}, block_h={largest.block_h})")
if oom:
    smallest = min(oom, key=asked)
    print(f"smallest that failed:   {asked(smallest) / 1024**2:.2f} MiB "
          f"(block_t={smallest.block_t}, block_h={smallest.block_h})")
    print()
    print("first failure, verbatim:")
    print(smallest.error[:800])
else:
    print("Nothing ran out of VMEM: the sweep did not reach the budget. The bound")
    print("is only that the default exceeds the largest working set above -- say")
    print("that, rather than reporting the largest config as the limit.")


### 4b · Does `vmem_limit_bytes` move the boundary?

If section 4 found a config the budget refused, re-run it with explicit limits. A config that
fails at the default and succeeds at a larger explicit limit proves two things at
once: the flag governs the budget, and the default is below the value that
worked. If nothing failed, this cell walks the limit *down* instead, until the
headline geometry stops compiling — which brackets the default from the other
side.

A second ladder then runs `block_h=6912`, where the single-buffer rule and the
flat double-buffer model disagree by a factor of two — enough to put the same
geometry on opposite sides of the chip's 128 MiB.

In [ ]:
MiB = 1024**2
LIMITS = [16 * MiB, 32 * MiB, 64 * MiB, 100 * MiB, 128 * MiB]

if oom:
    first_refused = min(oom, key=asked)
    probe_t, probe_h = first_refused.block_t, first_refused.block_h
    print(f"probing the first config the budget refused: block_t={probe_t}, "
          f"block_h={probe_h}, "
          f"{scoped_bytes(TOKENS, probe_t, probe_h) / MiB:.2f} MiB scoped")
else:
    probe_t, probe_h = BLOCK_T, BLOCK_H
    print(f"nothing ran out of VMEM at the default; walking the limit down on the "
          f"headline geometry: block_t={probe_t}, block_h={probe_h}, "
          f"{scoped_bytes(TOKENS, probe_t, probe_h) / MiB:.2f} MiB scoped")
print()


def limit_ladder(block_t, block_h):
    return [
        dict(tokens=TOKENS, block_t=block_t, block_h=block_h, precision=None,
             vmem_limit_bytes=limit)
        for limit in LIMITS
    ]


def keep_limit(result):
    vmem_results.append(result)
    bench.save(vmem_results, vmem_path)
    median = result.median()
    timing = f"{1e3 * median:8.3f} ms" if median else "        --"
    print(f"vmem_limit={result.vmem_limit_bytes / MiB:6.0f} MiB  "
          f"{failure_kind(result):<9} {timing}")


bench.sweep(limit_ladder(probe_t, probe_h), interpret=False, on_result=keep_limit)

# The single-buffer rule makes a claim sharp enough to be wrong. block_h=6912
# leaves the h axis one step long, so the 91.12 MiB of weights is allocated once
# rather than twice and the kernel asks 93.38 MiB -- more than a 64 MiB limit
# allows, less than a 100 MiB one. A flat "everything is double buffered" model
# puts the same geometry at 184.50 MiB, which fits nowhere on a 128 MiB chip and
# would have predicted a failure this ladder is about to contradict.
FULL_H = BLOCK_HS[-1]
print()
print(f"single-buffer test: block_t=128, block_h={FULL_H} asks "
      f"{scoped_bytes(TOKENS, 128, FULL_H) / MiB:.2f} MiB scoped, against "
      f"{info.vmem_capacity_bytes / MiB:.0f} MiB on the chip. Expect ok at 100 "
      f"and 128, refused below.")
print()
bench.sweep(limit_ladder(128, FULL_H), interpret=False, on_result=keep_limit)


## 5 · Token sweep — the roofline points

`T in {128, 256, 512, 1024, 2048}`, kernel and `jax.jit(reference.gated_mlp)` at
each, at `DEFAULT` and `HIGHEST` — the two precisions Mosaic will lower
(section 3).

`block_t` is pinned at `BEST_BLOCK_T` for every row, so only `T=128` is a single
`t` step. Every larger sequence walks `T // block_t` of them and re-reads the
whole 95.6 MB weight set on each, up to 1.53 GB at `T=2048` against the 114 MB
the problem actually requires. That is deliberate: holding the block fixed while
`T` grows is what isolates the re-streaming cost, and section 5b then removes
it. Read these rows as one kernel geometry measured at five sizes, not as the
kernel's best time at each size.

Warmup 3, repeats 20, `block_until_ready` on every call, and the whole
distribution kept rather than a mean: a bimodal set of samples is a fact about
the run.


In [ ]:
BEST_BLOCK_T = BLOCK_T
BEST_BLOCK_H = BLOCK_H  # set these from cell 4 if a larger block compiled

TOKEN_SWEEP = [128, 256, 512, 1024, 2048]

# DEFAULT and HIGHEST: 1 bf16 pass and 6. HIGH -- the 3-pass roof, and the one
# crossover this sweep could have resolved -- is not lowerable by Mosaic on jax
# 0.11.0 (cell 3). `bench.sweep` would record those rows as status="failed"
# rather than raise, but a column of known failures is not a measurement, so
# they are not run. HIGHEST is swept in its place: it costs nothing extra to
# collect and it puts the kernel against a second roof.
SWEEP_PRECISIONS = [None, "highest"]

token_specs = []
for precision in SWEEP_PRECISIONS:
    for tokens in TOKEN_SWEEP:
        token_specs.append(
            dict(
                tokens=tokens,
                block_t=min(tokens, BEST_BLOCK_T),
                block_h=BEST_BLOCK_H,
                precision=precision,
            )
        )
        token_specs.append(dict(label="xla", tokens=tokens, precision=precision))

token_results = []
token_path = RESULTS / "token_sweep.json"


def keep_token(result):
    token_results.append(result)
    bench.save(token_results, token_path)
    median = result.median()
    timing = f"{1e3 * median:8.3f} ms" if median else "        --"
    print(f"{result.label:<7} T={result.tokens:<5} {result.precision:<8} "
          f"{result.status:<7} {timing}")


bench.sweep(token_specs, interpret=False, on_result=keep_token)

### 5b · `block_t` — the term that actually moves bytes

Section 5 leaves the kernel 3x behind XLA at `DEFAULT` and level with it at
`HIGHEST`, which looks like two findings and is one. Per token on the
`1024 -> 2048` leg the kernel costs 1.04 us at `DEFAULT` against XLA's 0.267,
while the MLP's arithmetic is `6 * E * H = 47.8` MFLOP/token — 0.242 us at the
bf16 roof. XLA is running at 91% of the compute roof. The kernel is nowhere near
it, and 1.04 us/token is not arithmetic at all: it is 95.6 MB of weights fetched
once per `t` step at 718 GB/s, 88% of the *bandwidth* roof. The kernel is not
executing badly. It is moving eight times the bytes it needs to.

`HIGHEST` costs six bf16 passes, so the compute roof rises to 1.455 us/token and
swallows the 1.04. The kernel's 5% edge there is the bar dropping onto its
floor, not the kernel improving.

`bytes = (T // block_t) * 3 * E * H * 4 + 2 * T * E * 4` has exactly one term
under our control, and section 4 established that `block_h` is not it. So hold
`T=2048` and `DEFAULT`, vary `block_t`, and the model makes a shape rather than a
direction:

| `block_t` | bytes | roof | binds |
|---|---|---|---|
| 128 | 1548 MB | 1890 us | memory |
| 256 | 783 MB | 956 us | memory |
| 512 | 401 MB | 497 us | compute |
| 1024 | 210 MB | 497 us | compute |
| 2048 | 114 MB | 497 us | compute |

Halving the traffic halves the time until `block_t=512`, where the memory roof
crosses under the compute roof at 497 us and the curve goes flat. A kernel that
keeps improving past 512 falsifies the byte model; one that never improves
falsifies the diagnosis.

The largest geometry asks 28.13 MiB of scoped VMEM — `block_t=2048` makes the
`t` axis one step, so `x` and the output are single-buffered by the section 4
rule. Over the 16 MiB default, far under the chip, so `vmem_limit_bytes` is
raised to 64 MiB for every row rather than only the ones that need it: a limit
that varies across a sweep is a second variable.

XLA at `T=2048, DEFAULT` measured 0.778 ms in section 5, and it reads the
weights once. That is the number to beat.


In [ ]:
from gemma3_pallas.shapes import V5E, mlp_bytes, mlp_flops, roofline_bound

BLOCK_T_SWEEP = [128, 256, 512, 1024, 2048]
BLOCK_T_TOKENS = 2048
BLOCK_T_LIMIT = 64 * MiB

# DEFAULT is 1 bf16 pass and HIGHEST is 6, measured in section 3. The roof an
# fp32-typed kernel runs against is the bf16 roof divided by that count.
PASSES_FOR = {None: 1, "highest": 6}


def roof(tokens, block_t, precision):
    """(seconds at the roof, which roof binds) for one geometry."""
    return roofline_bound(
        mlp_flops(tokens),
        mlp_bytes(tokens, block_t=block_t),
        peak_flops=V5E.peak_flops(PASSES_FOR[precision]),
    )


block_t_specs = [
    dict(
        tokens=BLOCK_T_TOKENS,
        block_t=block_t,
        block_h=BEST_BLOCK_H,
        precision=None,
        vmem_limit_bytes=BLOCK_T_LIMIT,
    )
    for block_t in BLOCK_T_SWEEP
]
block_t_specs.append(dict(label="xla", tokens=BLOCK_T_TOKENS, precision=None))

block_t_results = []
block_t_path = RESULTS / "block_t_sweep.json"


def keep_block_t(result):
    block_t_results.append(result)
    bench.save(block_t_results, block_t_path)
    # XLA has no block; it reads the weights once, which is the same traffic as
    # the block_t = tokens geometry, so the model applies to its row unchanged.
    block_t = result.block_t or BLOCK_T_TOKENS
    moved = mlp_bytes(BLOCK_T_TOKENS, block_t=block_t)
    best, bound = roof(BLOCK_T_TOKENS, block_t, None)
    median = result.median()
    timing = f"{1e3 * median:7.3f} ms" if median else "      --"
    reached = f"{100 * best / median:5.1f}%" if median else "   --"
    print(f"{result.label:<7} block_t={block_t:<5} {moved / 1e6:7.1f} MB  "
          f"roof {1e6 * best:6.0f} us {bound:<7} {timing}  {reached} of roof")


print(f"T={BLOCK_T_TOKENS}, block_h={BEST_BLOCK_H}, DEFAULT, "
      f"vmem_limit={BLOCK_T_LIMIT // MiB} MiB")
print(f"largest ask: block_t={BLOCK_T_SWEEP[-1]} wants "
      f"{scoped_bytes(BLOCK_T_TOKENS, BLOCK_T_SWEEP[-1], BEST_BLOCK_H) / MiB:.2f}"
      f" MiB scoped")
print()

bench.sweep(block_t_specs, interpret=False, on_result=keep_block_t)


## 6 · Profiler — the copy-elision measurement

Trace about five iterations at the headline geometry and count the `w_*` DMAs.
The two byte models disagree by exactly a factor of `tokens // block_t`:

* weights re-read per `t` step → `tokens // block_t` transfers per weight
* copy elision across the inner-index reset → 1 transfer per weight

At `T=256, block_t=128` that is 2 versus 1, and the trace settles it. This is
also the highest-value single use of TPU time on the standing question of whether
copy elision is contractual on the hardware path — the docs state it as a
property, but the `pallas_call` pipeline on TPU is emitted by Mosaic, out of
reach from Python.

In [ ]:
import jax.profiler

from gemma3_pallas.shapes import arithmetic_intensity, mlp_bytes, mlp_flops


def headline(x, w_gate, w_up, w_down):
    return fused_gated_mlp(
        x, w_gate, w_up, w_down,
        block_t=BEST_BLOCK_T, block_h=BEST_BLOCK_H, interpret=False, precision=None,
    )


traced = jax.jit(headline)
jax.block_until_ready(traced(x, w_gate, w_up, w_down))  # compile outside the trace

with jax.profiler.trace(str(TRACES)):
    for _ in range(5):
        jax.block_until_ready(traced(x, w_gate, w_up, w_down))

passes = TOKENS // BEST_BLOCK_T
per_weight = cfg.embed_dim * cfg.hidden_dim * 4
print(f"trace written to {TRACES}")
print()
print(f"grid ({TOKENS // BEST_BLOCK_T}, {cfg.hidden_dim // BEST_BLOCK_H}), "
      f"{cfg.hidden_dim // BEST_BLOCK_H} hidden blocks per t step")
print(f"predicted w_* DMAs per weight, per call: {passes} "
      f"(elision would give 1)")
print(f"total weight bytes, {passes}-pass model: "
      f"{3 * passes * per_weight / 1e6:.1f} MB")
print(f"total weight bytes, elided model:        {3 * per_weight / 1e6:.1f} MB")
print()
for elide in (False, True):
    moved = mlp_bytes(TOKENS, block_t=BEST_BLOCK_T, elide_weights=elide)
    intensity = arithmetic_intensity(mlp_flops(TOKENS), moved)
    print(f"elide_weights={str(elide):<5} bytes={moved / 1e6:7.1f} MB  I={intensity:6.2f}")
print()
print("Count the w_gate/w_up/w_down HBM-to-VMEM transfers per call in the trace")
print("and compare. Re-open it locally with tensorboard-plugin-profile.")

## 7 · Summary and roofline plot

`PASSES` selects the compute roof and has no default anywhere in the library —
inventing one is exactly what `roofline_bound`'s required keyword exists to
prevent. Set it from cell 3: 1 for `DEFAULT`, 6 for `HIGHEST`. The 3-pass roof
is drawn for reference and **dashed**, because no row here ran at `HIGH`.

A point within 2% of its ridge prints as **at ridge**, not as a verdict. That is
not a hedge added after seeing the data: `DEFAULT` at T=512 was registered as
unresolvable before the run, and `HIGH`'s crossover — the one with a 21% margin —
was ruled out by the toolchain before any timing was collected.

In [ ]:
import matplotlib.pyplot as plt

from gemma3_pallas.shapes import V5E

PASSES = 1  # 1 = DEFAULT, 6 = HIGHEST -- read section 3 before setting (3 = HIGH
            # is not reachable on this jax, so no row here can be scored at it)

default_rows = [r for r in token_results if r.precision == "DEFAULT"]
highest_rows = [r for r in token_results if r.precision == "HIGHEST"]

print(bench.summarise(default_rows, passes=1))
print()
print(bench.summarise(highest_rows, passes=6))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

intensities = [2**i for i in range(0, 12)]

# All three roofs are drawn, but the 3-pass one is dashed and labelled as
# untested: nothing measured here ran at HIGH, and a solid line would invite the
# eye to read points against a roof no point belongs to.
ROOFS = (
    (1, "DEFAULT (1 bf16 pass)", "-"),
    (3, "HIGH (3) — not lowerable on this jax", "--"),
    (6, "HIGHEST (6)", "-"),
)
for passes, name, style in ROOFS:
    peak = V5E.peak_flops(passes)
    roof = [min(peak, i * V5E.peak_hbm_bandwidth) / 1e12 for i in intensities]
    ax.plot(intensities, roof, style, label=f"{name} — ridge {V5E.ridge_point(passes):.0f}")

for rows, passes, marker in ((default_rows, 1, "o"), (highest_rows, 6, "s")):
    xs, ys = [], []
    for r in rows:
        if r.status != "ok" or r.block_t is None:
            continue
        flops = mlp_flops(r.tokens)
        xs.append(arithmetic_intensity(flops, mlp_bytes(r.tokens, block_t=r.block_t)))
        ys.append(flops / r.median() / 1e12)
    if xs:
        ax.scatter(xs, ys, marker=marker, zorder=3, label=f"kernel, {passes} pass(es)")

ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.set_xlabel("arithmetic intensity (FLOP/byte)")
ax.set_ylabel("achieved (TFLOP/s)")
ax.set_title("fused_gated_mlp on TPU v5e — three roofs, two measured")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RESULTS / "roofline.png", dpi=150)
plt.show()

## 8 · Package

`results/` and `traces/` live on the runtime's disk and go when it does, so this
cell bundles them into one zip and downloads it through the browser. Worth it for
the trace in particular: re-opening it locally with `tensorboard-plugin-profile`
is how the DMA count gets read.

The numbers reach the repo as prose in
[`docs/measurements/0001-fused-gated-mlp-on-v5e.md`](https://github.com/denis-mil/gemma3-tpu-pallas/blob/main/docs/measurements/0001-fused-gated-mlp-on-v5e.md);
the raw artifacts are gitignored.

In [ ]:
import datetime
import shutil

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
staging = pathlib.Path(f"/content/phase2_v5e-{stamp}")
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(RESULTS, staging / "results")
shutil.copytree(TRACES, staging / "traces")

archive = pathlib.Path(shutil.make_archive(str(staging), "zip", root_dir=staging))
shutil.rmtree(staging, ignore_errors=True)

print(archive, f"{archive.stat().st_size / 1e6:.1f} MB")
for path in sorted(RESULTS.rglob("*")):
    if path.is_file():
        print(f"  results/{path.relative_to(RESULTS)}  {path.stat().st_size / 1e3:.0f} kB")
trace_files = [path for path in TRACES.rglob("*") if path.is_file()]
print(f"  traces/  {len(trace_files)} file(s), "
      f"{sum(path.stat().st_size for path in trace_files) / 1e6:.1f} MB")

from google.colab import files

files.download(str(archive))